# Laboratorio de introducción · Parte 3 (opcional): minianálisis con pandas

**Asignatura:** Ciencia de Datos · Grado en Biomedicina (UAX), 0431304

---

Esta parte es **opcional**: no se entrega y no se evalúa. Repite el flujo de la Parte 2 (cargar,
inspeccionar, resumir y visualizar) con pandas escrito como se usa en el trabajo diario, encadenando
operaciones.

Es el recorrido guiado de la ampliación opcional de la guía, con el código ordenado y con
comprobaciones. Si prefieres escribir tu propio cuaderno desde cero con los fragmentos de la guía (es
lo que más enseña), hazlo primero y usa este después para comparar.

**Qué necesitas:** este cuaderno y el archivo `cohorte_neumonia.csv`, la misma cohorte SIMULADA de la
Parte 2, con fines docentes (no son pacientes reales). En Colab, súbelo antes con el panel
**Archivos** (icono de carpeta), como en la Parte 2.

Las respuestas deben coincidir con las del script de R de esta misma Parte 3.

## Bloque 0. Importar las bibliotecas

En Colab, pandas, numpy y matplotlib ya vienen instaladas: solo hay que importarlas, una vez por sesión.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

> **Interpretación de resultados**
>
> La celda no imprime nada: si no sale ningún error, las bibliotecas están listas. Si sale
> `ModuleNotFoundError`, falta instalar esa biblioteca (en Colab, desde una celda con
> `!pip install nombre`).

## Bloque 1. Cargar el dataset

`pd.read_csv()` lee el CSV y lo convierte en un DataFrame. Se prueba primero la ruta de la carpeta del
laboratorio y, si no existe, la de Colab.

In [ ]:
archivo = "../data/cohorte_neumonia.csv"      # en tu ordenador: carpeta data, un nivel por encima
if not os.path.exists(archivo):
    archivo = "cohorte_neumonia.csv"          # en Colab: sube antes el CSV (panel Archivos, icono de carpeta)
if not os.path.exists(archivo):
    raise FileNotFoundError("No encuentro cohorte_neumonia.csv. En Colab, súbelo desde el panel "
                            "Archivos y vuelve a ejecutar esta celda.")
cohorte = pd.read_csv(archivo)

print(cohorte.shape)   # (filas, columnas)
cohorte.info()         # una línea por variable: nombre, valores no vacíos y tipo
cohorte.head()         # las primeras filas

> **Interpretación de resultados**
>
> 1.800 filas × 21 columnas: la tabla tal como llega, sin ninguna columna derivada. `info()` muestra el
> tipo de cada columna (`int64` y `float64` números, `object` o `str` texto). Las fechas se leen como
> texto: pandas no las convierte a fecha si no se le pide.

## Bloque 2. Primera exploración descriptiva

Contáis el desenlace y resumís edad, sexo y comorbilidades.

In [ ]:
print(cohorte["ingreso_uci"].value_counts())
print("Prevalencia de UCI:", round(cohorte["ingreso_uci"].mean(), 3))
print("Edad media:", round(cohorte["edad"].mean(), 1), "· mediana:", cohorte["edad"].median())
print(cohorte["sexo"].value_counts())
print(cohorte[["diabetes", "epoc", "insuf_cardiaca", "hipertension", "ins_renal"]].sum())

> **Interpretación de resultados**
>
> `ingreso_uci`: 1.395 pacientes con 0 (no) y 405 con 1 (sí), una prevalencia de 0,225 (22,5 %). Edad
> media 66,6 y mediana 65. Sexo: 979 H y 821 M. La hipertensión es la comorbilidad más frecuente.
>
> Esa prevalencia de 0,225 reaparece en los laboratorios: es la misma cohorte la que acompaña a toda la
> asignatura.

## Bloque 3. Filtrar, crear, agrupar y resumir

Son las operaciones que usaréis siempre. Tienen equivalente directo en el tidyverse de R: filtrar filas
(`query()` o `df[condición]`, como `filter()`), crear variables nuevas (`pd.cut()` o `assign()`,
como `mutate()`) y agrupar y resumir (`groupby()` + `agg()`, como `group_by()` + `summarise()`).

In [ ]:
# Filtrar: los pacientes de 65 o más años.
mayores = cohorte.query("edad >= 65")
print(len(mayores))

# Crear una variable con tramos de edad y contar cuántos hay en cada uno.
# (Se guarda aparte, sin añadirla a la tabla, igual que el mutate() del script de R.)
grupo_edad = pd.cut(cohorte["edad"], bins=[0, 49, 64, 79, np.inf],
                    labels=["<50", "50-64", "65-79", "80+"])
print(grupo_edad.value_counts().sort_index())

# Agrupar y resumir: PCR media según el desenlace.
print(cohorte.groupby("ingreso_uci")["pcr_mg_l"].mean().round(1))

pandas también admite un estilo **encadenado** (*method chaining*): cada línea es una operación que
recibe el resultado de la anterior, igual que el *pipe* `|>` de R. Es el ejemplo con el que la guía
presenta el apartado, leído en voz alta: «coge la cohorte, quédate con los de 65 o más años, sepáralos
por ingreso en UCI y calcula para cada grupo la PCR media y el número de pacientes».

In [ ]:
(cohorte
   .query("edad >= 65")
   .groupby("ingreso_uci")
   .agg(media_pcr=("pcr_mg_l", "mean"), n=("pcr_mg_l", "size"))
   .round(1)
   .reset_index())

> **Interpretación de resultados**
>
> 954 pacientes tienen 65 años o más. Por tramos de edad: <50, 193; 50-64, 653; 65-79, 616; 80+, 338
> (suman 1.800). PCR media por desenlace: aproximadamente 85,6 (no UCI) frente a 114,5 (UCI). Los que
> van a UCI llegan con más inflamación: ya aquí se intuye que la PCR separa a los grupos.

## Bloque 4. Primeros gráficos

Un histograma de edad y un boxplot de PCR por desenlace, con las funciones de dibujo de pandas (que
usan matplotlib por debajo).

In [ ]:
cohorte["edad"].plot.hist(bins=range(25, 105, 5), edgecolor="white") #plot.hist() es un método de pandas, que a su vez llama a matplotlib. bins=range(25, 105, 5) indica los límites de los intervalos (de 25 a 100 años, de 5 en 5). edgecolor="white" dibuja un borde blanco alrededor de cada barra.
plt.title("Edad de la cohorte")
plt.xlabel("Edad (años)")
plt.ylabel("Pacientes")
plt.show()

cohorte.boxplot(column="pcr_mg_l", by="ingreso_uci") #boxplot() es un método de pandas, que a su vez llama a matplotlib. column indica la variable que se dibuja, by indica la variable que define los grupos (cada grupo tendrá su propia caja). El resultado es un gráfico de cajas y bigotes (boxplot) de PCR según ingreso en UCI.
plt.suptitle("")   # quita el título automático que añade pandas
plt.title("PCR al ingreso según ingreso en UCI")
plt.xlabel("Ingreso en UCI (0 = no, 1 = sí)")
plt.ylabel("PCR (mg/L)")
plt.show()

> **Interpretación de resultados**
>
> El histograma de edad tiene su pico entre los 55 y los 70 años y una cola hacia edades altas; la
> última barra sube porque hay 48 pacientes con 99 años exactos (el tope de la simulación). En el
> boxplot, la caja de PCR del grupo UCI queda más alta: el gráfico comunica de un golpe lo que luego confirmarán los contrastes estadísticos.

## Bloque 5. Mini-EDA: cuatro preguntas y comprobación

1. ¿Cuántos pacientes hay y cuántas variables tiene el dataset?
2. ¿Cuál es la edad media y mediana de la cohorte?
3. ¿Qué porcentaje de pacientes ingresó en UCI?
4. ¿La PCR al ingreso difiere visualmente entre los que ingresan en UCI y los que no? Es el boxplot del
   Bloque 4.

In [ ]:
print("1)", len(cohorte), "pacientes y", cohorte.shape[1], "variables")
print("2) Edad media", round(cohorte["edad"].mean(), 1), "y mediana", cohorte["edad"].median())
print("3)", round(100 * cohorte["ingreso_uci"].mean(), 1), "% ingresó en UCI")

assert len(cohorte) == 1800, "La cohorte debería tener 1.800 filas"
assert cohorte.shape[1] == 21, "La cohorte debería tener 21 variables"
assert cohorte["ingreso_uci"].sum() == 405, "Debería haber 405 ingresos en UCI"
assert round(cohorte["edad"].mean(), 1) == 66.6, "La edad media no cuadra"
assert cohorte["edad"].median() == 65, "La mediana de edad no cuadra"
assert len(mayores) == 954, "Debería haber 954 pacientes de 65 o más años"

print("CHECKPOINT SUPERADO: 1.800 pacientes y 21 variables, edad media 66,6")
print("(mediana 65), 22,5 % en UCI y 954 pacientes de 65 o más años.")

> **Interpretación de resultados**
>
> Lo importante no es el resultado en sí, sino el flujo: cargar datos, inspeccionar, resumir y
> visualizar. Es el patrón que se repite en cada laboratorio de la asignatura, con preguntas más
> complejas y técnicas más avanzadas, pero con la misma estructura mental. Si algún `assert` se
> detiene, casi siempre significa que se ha leído un archivo distinto del que toca.

---

### Fin de la Parte 3 (opcional)

Si has llegado hasta aquí sin errores, dominas el flujo básico con pandas encadenado: es la forma de
escribir que encontrarás en casi todo el código de análisis de datos en Python.